# 09 · Mixture-of-Experts: router, dispatch and load balancing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/09_moe_router.ipynb) &nbsp; 📖 Concept deck: [Mixture of Experts](https://nilesh-salpe.github.io/visual-ai/architecture/10-mixture-of-experts/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** Mixtral-style top-k routing, a sparse MoE layer, the Switch Transformer load-balancing loss, and expert capacity with token dropping.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

r = np.random.default_rng(0)
d, n_exp, h = 16, 8, 32
experts = [(r.normal(0, d ** -.5, (d, h)), r.normal(0, h ** -.5, (h, d))) for _ in range(n_exp)]
expert_fn = lambda e, x: np.maximum(0, x @ experts[e][0]) @ experts[e][1]
W_g = r.normal(0, d ** -.5, (d, n_exp))

## Task 1: Top-k routing (Mixtral)

Mixtral: $G(x) = \mathrm{Softmax}(\mathrm{TopK}(x\cdot W_g))$. Keep the $k$ largest router logits per token and softmax over **only those $k$**. The other experts get weight 0.
Return `idx` [T, k] (expert ids, largest logit first) and `w` [T, k] (weights summing to 1 per token).

In [ ]:
def topk_route(logits, k):
    """logits: [T, n_exp]. Returns (idx [T, k], w [T, k])."""
    # TODO: argsort descending, take the first k, softmax over those k logits
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_topk_route(f):
    lg = np.array([[1.0, 3.0, 2.0, -1.0], [0.0, 0.0, 5.0, 4.0]])
    idx, w = f(lg, 2)
    assert idx.shape == (2, 2) and w.shape == (2, 2), "shapes must be [T, k]"
    assert idx.tolist() == [[1, 2], [2, 3]], f"expected [[1, 2], [2, 3]], got {idx.tolist()}"
    assert np.allclose(w.sum(1), 1), "weights per token must sum to 1"
    assert np.allclose(w[0], softmax(np.array([3.0, 2.0]))), "softmax over the top-k logits only"

check("topk_route", _test_topk_route, topk_route)

## Task 2: The sparse MoE layer

$y = \sum_{i \in \text{top-}k} G(x)_i \cdot E_i(x)$. Each token runs only $k$ of the $n$ experts. For Mixtral: 8 experts, $k=2$, 47B total params, about 13B active per token.
Use `expert_fn(e, x)` from setup. Make it **sparse**: for each expert, gather only the tokens routed to it and run the expert once on that batch. This dispatch pattern becomes all-to-all communication under expert parallelism.

In [ ]:
def moe_layer(x, k=2):
    """x: [T, d]. Returns (y [T, d], idx [T, k])."""
    # TODO: route with topk_route(x @ W_g, k); for each expert e, find (token, slot) pairs routed to e,
    #       run expert_fn(e, x[tokens]) once, add weight * output into y[tokens]
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_moe_layer(f):
    x = np.random.default_rng(1).normal(size=(12, d))
    y, idx = f(x, 2)
    ref_idx, ref_w = topk_route_solution(x @ W_g, 2)
    ref = np.stack([sum(ref_w[t, j] * expert_fn(ref_idx[t, j], x[t:t + 1])[0] for j in range(2)) for t in range(12)])
    assert y.shape == (12, d)
    assert np.allclose(y, ref, atol=1e-10), "output must equal the weighted sum of each token's top-k experts"
    calls = []
    global expert_fn
    orig = expert_fn; expert_fn = lambda e, xx: (calls.append(e), orig(e, xx))[1]
    try: f(x, 2)
    finally: expert_fn = orig
    assert len(calls) <= n_exp, f"call each expert at most once per layer (batched dispatch); got {len(calls)} calls"

check("moe_layer", _test_moe_layer, moe_layer)

## Task 3: Switch Transformer load-balancing loss

Left alone, a router collapses onto a few experts. Switch Transformer (eq. 4–6) adds
$$\text{loss} = \alpha\cdot N\cdot\sum_{i=1}^{N} f_i\,P_i$$
* $f_i$: the fraction of tokens whose **top-1** expert is $i$
* $P_i$: the mean router probability (full softmax) assigned to expert $i$
* $\alpha = 10^{-2}$ in the paper

Under perfectly uniform routing $f_i = P_i = 1/N$, so the loss is exactly $\alpha$. If every token picks one expert with certainty, the loss is $\alpha N$.

In [ ]:
def load_balance_loss(logits, alpha=1e-2):
    """logits: [T, N] router logits."""
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_load_balance_loss(f):
    T, N = 64, 8
    uni = np.tile(np.eye(N) * 1e-3, (T // N, 1))
    assert np.isclose(f(uni), 1e-2, rtol=1e-3), f"uniform routing should give alpha = 0.01, got {f(uni):.5f}"
    col = np.zeros((T, N)); col[:, 3] = 50
    assert np.isclose(f(col), 1e-2 * N, rtol=1e-3), f"total collapse should give alpha * N = 0.08, got {f(col):.5f}"
    r = np.random.default_rng(2); lg = r.normal(size=(T, N))
    p = softmax(lg); fi = np.bincount(lg.argmax(1), minlength=N) / T
    assert np.isclose(f(lg), 1e-2 * N * (fi * p.mean(0)).sum()), "use top-1 fractions f_i and mean full-softmax probabilities P_i"

check("load_balance_loss", _test_load_balance_loss, load_balance_loss)

## Task 4: Expert capacity and dropped tokens

On accelerators each expert gets a fixed-size buffer:
$$\text{capacity} = \frac{\text{tokens per batch}}{\text{number of experts}} \times \text{capacity factor}$$
Tokens routed to a full expert are **dropped** (they skip the layer through the residual connection).
Given top-1 assignments `assign` [T], return a boolean `kept` [T]: keep tokens in arrival order until their expert's buffer (`int(capacity)`) is full.

In [ ]:
def capacity_dispatch(assign, n_experts, capacity_factor):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 4: run after implementing
def _test_capacity_dispatch(f):
    a = np.array([0, 0, 0, 1, 0, 1, 2, 3])
    k = f(a, 4, 1.0)          # capacity = 8/4 * 1.0 = 2
    assert k.tolist() == [True, True, False, True, False, True, True, True], f"got {k.tolist()}"
    assert f(a, 4, 2.0).all(), "capacity 4 fits everything"

check("capacity_dispatch", _test_capacity_dispatch, capacity_dispatch)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Top-k routing (Mixtral)
def topk_route_solution(logits, k):
    idx = np.argsort(-logits, axis=1)[:, :k]
    top = np.take_along_axis(logits, idx, axis=1)
    return idx, softmax(top, axis=1)

check("topk_route (reference)", _test_topk_route, topk_route_solution)

In [ ]:
# @title Solution for Task 2: The sparse MoE layer
def moe_layer_solution(x, k=2):
    idx, w = topk_route_solution(x @ W_g, k); y = np.zeros_like(x)
    for e in range(n_exp):
        tok, slot = np.nonzero(idx == e)
        if len(tok):
            y[tok] += w[tok, slot][:, None] * expert_fn(e, x[tok])
    return y, idx

check("moe_layer (reference)", _test_moe_layer, moe_layer_solution)

In [ ]:
# @title Solution for Task 3: Switch Transformer load-balancing loss
def load_balance_loss_solution(logits, alpha=1e-2):
    T, N = logits.shape
    f = np.bincount(logits.argmax(1), minlength=N) / T
    P = softmax(logits, axis=1).mean(0)
    return alpha * N * (f * P).sum()

check("load_balance_loss (reference)", _test_load_balance_loss, load_balance_loss_solution)

In [ ]:
# @title Solution for Task 4: Expert capacity and dropped tokens
def capacity_dispatch_solution(assign, n_experts, capacity_factor):
    cap = int(len(assign) / n_experts * capacity_factor); used = np.zeros(n_experts, int); kept = np.zeros(len(assign), bool)
    for t, e in enumerate(assign):
        if used[e] < cap:
            kept[t] = True; used[e] += 1
    return kept

check("capacity_dispatch (reference)", _test_capacity_dispatch, capacity_dispatch_solution)

---
## Experiments
**Total vs active parameters, and what balancing buys.** First, the arithmetic of sparsity using Mixtral's published figures. Then route 4,096 tokens through a deliberately skewed router and see how the load-balancing loss and the drop rate respond as the skew shrinks. That shrinking is what the auxiliary loss pushes training toward.

In [ ]:
print("Mixtral 8x7B: 8 experts, top-2 -> 47B total parameters, ~13B active per token")
print("DeepSeek-V2: 2 shared + 160 routed experts, 6 routed active -> 236B total, 21B active\n")
r = np.random.default_rng(0); T, N = 4096, 8
for skew in [3.0, 1.5, 0.5, 0.0]:
    lg = r.normal(size=(T, N)) + skew * np.linspace(1, 0, N)
    a = lg.argmax(1); kept = capacity_dispatch_solution(a, N, 1.25)
    load = np.bincount(a, minlength=N) / T
    print(f"skew {skew:3.1f}: busiest expert gets {load.max():.0%} of tokens, aux loss {load_balance_loss_solution(lg):.4f}, "
          f"dropped at capacity factor 1.25: {1 - kept.mean():.1%}")

---
## Takeaways
* MoE replaces the dense FFN with $N$ expert FFNs and a router. Each token uses $k$ of them, so parameters grow with $N$ while FLOPs grow with $k$.
* Without an auxiliary loss, routing collapses. The Switch loss $\alpha N\sum f_iP_i$ is minimized by uniform routing.
* Fixed expert capacity makes compute static but drops overflow tokens. Load balance is therefore a quality issue as well as an efficiency one.